# Credit Card Fraud Detection

End-to-end exploration, model comparison, threshold tuning, evaluation, and inference.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

In [ ]:
cd=pd.read_csv("../data/creditcard.csv")
cd.head(50)

In [ ]:
cd['Class'].value_counts(normalize=True)

In [ ]:
cd.isnull().sum()

In [ ]:
cd.info()

In [ ]:
cd.describe()

In [ ]:
cd.isnull().sum()

In [ ]:
cd.shape

In [ ]:
cd["Class"].value_counts()

In [ ]:
cd['Class'].value_counts(normalize=True)*100

In [ ]:
cd['Class'].value_counts(normalize=True)*100

In [ ]:
cd['Class'].value_counts().plot(kind="bar")
plt.xlabel("Class")
plt.ylabel("Number of transctions")
plt.title("Genuine vs Fraud Transctions")
plt.show()

In [ ]:
X=cd.drop("Class",axis=1)
y=cd['Class']

In [ ]:
print(X.shape)
print(y.shape)

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
print("X_train",X_train.shape)
print("X_test",X_test.shape)
print("y_train",y_train.shape)
print("y_test",y_test.shape)

In [ ]:
print(y_train.value_counts())
print(y_test.value_counts())

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
cols=['Time','Amount']
X_train_scaled=X_train.copy()
X_test_scaled=X_test.copy()

In [ ]:
X_train_scaled[cols]=scaler.fit_transform(X_train_scaled[cols])
X_test_scaled[cols]=scaler.transform(X_test_scaled[cols])

In [ ]:
print(X_train[cols].head())
print(X_train_scaled[cols].head())

In [ ]:
model=LogisticRegression(
    class_weight="balanced",
    max_iter=1000)
model.fit(X_train_scaled,y_train)

In [ ]:
print(model.classes_)
print(model.n_features_in_)


In [ ]:
y_pred=model.predict(X_test_scaled)
print(y_pred[:10])

In [ ]:
from sklearn.metrics import confusion_matrix
cm=confusion_matrix(y_test, y_pred)
print(cm)

In [ ]:
tn,fp,fn,tp=cm.ravel()
print('tn',tn)
print('fp',fp)
print('fn',fn)
print('tp',tp)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["Genuine", "Fraud"],
    cmap="Blues"
)
plt.title("Credit Card Fraud Detection")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import classification_report 
print(classification_report(y_test,y_pred))

In [ ]:
# Remove exact duplicate rows for this experiment
data=cd.drop_duplicates().copy()
X=data.drop("Class", axis=1)
y=data["Class"]
# Separate final test data (20%)
X_dev, X_final, y_dev, y_final = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
# Split remaining data into training and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev,
    test_size=0.25,
    random_state=42,
    stratify=y_dev
)
print(X_train.shape)
print(X_val.shape)
print(X_final.shape)

In [ ]:

# Import Pipeline to combine preprocessing and ML model
from sklearn.pipeline import Pipeline

# Import ColumnTransformer to process selected columns
from sklearn.compose import ColumnTransformer

# STEP 1: Define which columns need scaling
preprocess = ColumnTransformer(
    [
        # "scale" = Name of this preprocessing step
        # StandardScaler() = Scales values using mean and std
        # ["Time", "Amount"] = Columns we want to scale
        ("scale", StandardScaler(), ["Time", "Amount"])
    ],

    # Keep all other columns (V1 to V28) unchanged
    remainder="passthrough"
)


# STEP 2: Create a Machine Learning Pipeline
lr = Pipeline([
    # First, automatically preprocess the input data
    ("preprocess", preprocess),

    # Then, use Logistic Regression to detect fraud
    ("classifier", LogisticRegression(

        # Give higher weight to rare fraud examples
        class_weight="balanced",

        # Allow a maximum of 1000 training iterations
        max_iter=1000
    ))
])


# STEP 3: Train the entire Pipeline
# X_train = Input features (30 columns)
# y_train = Correct answers (0 = Genuine, 1 = Fraud)
# .fit() = Fit scaler and train Logistic Regression

lr.fit(X_train, y_train)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=12,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

In [ ]:

# Import function to evaluate fraud detection performance
from sklearn.metrics import average_precision_score


# STEP 1: Store our already-trained ML models
# models = Dictionary containing model names and models
# lr = Trained Logistic Regression model
# rf = Trained Random Forest model

models = {
    "Logistic Regression": lr,
    "Random Forest": rf
}


# STEP 2: Create an empty dictionary
# This will store the score of each model

scores = {}


# STEP 3: Compare both models one by one
# .items() = Gives the name and model from dictionary
# name = Name of current model
# model = Actual trained ML model

for name, model in models.items():

    # Predict fraud probability for validation transactions
    # X_val = Validation input features
    # predict_proba() = Predict probabilities of 0 and 1
    # [:, 1] = Select Class 1 (Fraud) probabilities
    probabilities = model.predict_proba(X_val)[:, 1]

    # Calculate Average Precision score
    # y_val = Actual answers (0 or 1)
    # probabilities = Predicted fraud probabilities
    # Higher AP = Better fraud ranking performance
    score = average_precision_score(
        y_val, probabilities
    )

    # Store the score under the model's name
    # Example: scores["Random Forest"] = 0.8488
    scores[name] = score

    # Display the model's name and score
    # round(score, 4) = Round to 4 decimal places
    print(name, ":", round(score, 4))


# STEP 4: Find which model has the highest score
# max() = Finds the maximum according to key
# scores.get = Gets the score of each model name
# best_name = Name of model with highest AP score

best_name = max(scores, key=scores.get)


# STEP 5: Get the actual best trained model
# models[best_name] = Retrieves model from dictionary
# best_model = Stores selected trained model

best_model = models[best_name]


# STEP 6: Display the winning model's name
print("Best model:", best_name)


In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve

val_prob = best_model.predict_proba(X_val)[:, 1]

precision, recall, thresholds = precision_recall_curve(
    y_val, val_prob
)

f1 = (
    2 * precision[:-1] * recall[:-1]
    / np.maximum(precision[:-1] + recall[:-1], 1e-12)
)

best_threshold = float(thresholds[np.argmax(f1)])

print("Chosen threshold:", best_threshold)

In [ ]:

# STEP 1: Import evaluation functions
from sklearn.metrics import (
    confusion_matrix,          # Shows correct and wrong predictions
    classification_report,     # Shows Precision, Recall, F1, Support
    average_precision_score    # Measures fraud-ranking performance
)


# STEP 2: Predict fraud probabilities on final test data
# best_model = Best model selected using validation data
# X_final = Final test input features (30 columns)
# predict_proba() = Predict probabilities for Class 0 and 1
# [:, 1] = Select only Class 1 (Fraud) probabilities

final_prob = best_model.predict_proba(X_final)[:, 1]


# STEP 3: Convert probabilities into Class 0 or 1
# best_threshold = Decision threshold chosen on validation data
# >= = Check if fraud probability reaches the threshold
# True = Fraud (1), False = Genuine (0)
# .astype(int) = Convert True/False into 1/0

final_pred = (final_prob >= best_threshold).astype(int)


# STEP 4: Display Confusion Matrix
# y_final = Actual correct answers
# final_pred = Model's predicted answers
# confusion_matrix() = Shows TP, TN, FP, FN

print("Confusion Matrix:")
print(confusion_matrix(y_final, final_pred))


# STEP 5: Display Classification Report
# Shows Precision, Recall, F1-score and Support
# zero_division=0 = Return 0 if a metric is undefined
# instead of showing a division warning

print("Classification Report:")
print(classification_report(
    y_final, final_pred, zero_division=0
))


# STEP 6: Calculate Average Precision
# y_final = Actual correct answers (0 or 1)
# final_prob = Predicted fraud probabilities
# average_precision_score() = Evaluates how well
# the model ranks fraud above genuine transactions

print(
    "Average Precision:",
    average_precision_score(y_final, final_prob)
)


In [ ]:

import joblib

# STEP 1: Load the saved file
saved = joblib.load("../models/creditcard_fraud_model.joblib")

# STEP 2: Extract the trained model
model = saved["model"]

# STEP 3: Extract the saved decision threshold
threshold = saved["threshold"]

# STEP 4: Extract input feature names
features = saved["features"]

# STEP 5: Prepare transactions in the saved feature order
# new_data must contain the required 30 feature columns
new_data = X_val[features]

# STEP 6: Predict fraud probabilities
probabilities = model.predict_proba(new_data)[:, 1]

# STEP 7: Convert probabilities into predictions
predictions = (probabilities >= threshold).astype(int)

# 0 = Genuine, 1 = Fraud
print(predictions[:10])


In [ ]:

# STEP 1: Import joblib to load our saved ML model
import joblib

# Load our previously saved model file
# Contains the trained model, threshold, and feature names
saved = joblib.load("../models/creditcard_fraud_model.joblib")


# STEP 2: Select one transaction for prediction
# X_final = Final test dataset
# .iloc[[0]] = Select first row as a DataFrame
# saved["features"] = Keep the 30 features in correct order

new_transaction = X_final.iloc[[0]][saved["features"]]


# STEP 3: Predict fraud probability
# saved["model"] = Our saved trained ML model
# predict_proba() = Predict probability for each class
# [0, 1] = First transaction, Class 1 (Fraud)
# The Pipeline automatically performs preprocessing

fraud_probability = saved["model"].predict_proba(
    new_transaction
)[0, 1]


# STEP 4: Convert probability into Class 0 or 1
# saved["threshold"] = Previously selected decision threshold
# >= = Check whether fraud probability reaches threshold
# True = Fraud (1), False = Genuine (0)
# int() = Convert True/False into 1/0

prediction = int(
    fraud_probability >= saved["threshold"]
)


# STEP 5: Display prediction
# 0 = Genuine transaction
# 1 = Fraudulent transaction

print("Prediction:", prediction)


# STEP 6: Display predicted fraud probability
# This is the model's fraud probability estimate
# A value of 0.85 represents an estimated 85% fraud probability

print("Model score:", fraud_probability)


In [ ]:

# STEP 1: Display the actual class
# y_final = Actual labels of final test transactions
# .iloc[0] = Select the first transaction's label
# 0 = Genuine, 1 = Fraud

print("Actual Class:", y_final.iloc[0])


# STEP 2: Display the predicted class
# prediction = Output from our trained model
# 0 = Genuine, 1 = Fraud

print("Predicted Class:", prediction)


# STEP 3: Compare actual and predicted class
# if = Checks whether a condition is True
# == = Checks whether two values are equal
# y_final.iloc[0] = Actual class of first transaction
# prediction = Class predicted by our ML model

if y_final.iloc[0] == prediction:

    # Runs when actual and predicted classes match
    print("Correct Prediction!")

else:

    # Runs when actual and predicted classes differ
    print("Wrong Prediction!")


In [ ]:

# STEP 1: Import model evaluation metrics
from sklearn.metrics import (

    # Accuracy = Percentage of all correct predictions
    accuracy_score,

    # Precision = Out of predicted frauds,
    # how many were actually fraud?
    precision_score,

    # Recall = Out of all actual frauds,
    # how many did our model detect?
    recall_score,

    # F1 = Combines Precision and Recall into one score
    f1_score
)


# STEP 2: Calculate Accuracy
# y_final = Actual correct labels (0 or 1)
# final_pred = Labels predicted by our model
# accuracy_score() = Correct / Total predictions

print("Accuracy:", accuracy_score(y_final, final_pred))


# STEP 3: Calculate Precision
# Checks how trustworthy our fraud predictions are
# Higher precision = Fewer false fraud alarms

print("Precision:", precision_score(y_final, final_pred))


# STEP 4: Calculate Recall
# Checks how many real fraud transactions we detected
# Higher recall = Fewer actual frauds missed

print("Recall:", recall_score(y_final, final_pred))


# STEP 5: Calculate F1 Score
# Combines Precision and Recall using harmonic mean
# High F1 = Good balance of precision and recall

print("F1 Score:", f1_score(y_final, final_pred))
